In [1]:
import warnings
import pandas as pd
import numpy as np

warnings.filterwarnings("ignore", category=UserWarning, module="openpyxl")

transaction_path = r"G:\My Drive\New_Working_Files\KNIME\CTU_2025\Payment_Adonc\Refund\Transaction\Transaction_refund.csv"


In [2]:
df = pd.read_csv(transaction_path, low_memory=False)

In [3]:
df.head()
print(df.columns)

Index(['refund_year', 'refund_month', 'refund_week', 'refund_init_date',
       'refund_complete_date', 'refund_status', 'refund_final_status_updated',
       'mr_refund_flag', 'refund_mode', 'auth_state', 'payment_mode',
       'instrument_type', 'payment_instrument', 'ref_amount_flag',
       'transaction_source', 'pg_id', 'promise_sla_refund_bucket',
       'refund_init_completion_bucket', 'overall_ageing_Stuck',
       'refund_complition_bucket', 'ARN_ISSUE_FLAG', 'void_flag',
       'refund_reason', 'refund_reason_flag', 'flipkart_emi_flag',
       'payment_type_final', 'cx_count', 'rf_orders', 'p_refund_count',
       'c_refund_count', 'total_refund_amount', 'is_shopsy_order',
       'marketplace_id'],
      dtype='object')


In [4]:
df=df[df['marketplace_id'] !="marketplace_id"]
df = df[df['refund_init_date'] !="refund_init_date"]
df['refund_init_date'] =df['refund_init_date'].astype("datetime64[ns]")
df['p_refund_count'] = pd.to_numeric(df['p_refund_count'], errors ='coerce')
df['is_shopsy_order'] = df['is_shopsy_order'].astype(str).str.lower()
df=df.sort_values(by=["refund_init_date"])
df[df['refund_init_date'].isna()]
print(df['is_shopsy_order'].unique())
print(df['marketplace_id'].unique())

['false' 'true' 'nan']
['FLIPKART' 'HYPERLOCAL' nan 'GROCERY' 'RCBP']


In [5]:
print(df.dtypes)


refund_year                               int64
refund_month                              int64
refund_week                               int64
refund_init_date                 datetime64[ns]
refund_complete_date                     object
refund_status                            object
refund_final_status_updated              object
mr_refund_flag                           object
refund_mode                              object
auth_state                               object
payment_mode                             object
instrument_type                          object
payment_instrument                       object
ref_amount_flag                          object
transaction_source                       object
pg_id                                    object
promise_sla_refund_bucket                object
refund_init_completion_bucket            object
overall_ageing_Stuck                     object
refund_complition_bucket                 object
ARN_ISSUE_FLAG                          

In [6]:
df= df[(~df["marketplace_id"].isin(["HYPERLOCAL"])) &
                          ~df["is_shopsy_order"].isin(["true"])].copy()


filtered_date = df[df['refund_init_date'].isin(['2026-01-10','2026-01-11','2026-01-12','2026-01-13'])]

refund_final_status =(
    filtered_date.groupby(['refund_init_date','refund_final_status_updated'],observed=False)["p_refund_count"]
    .sum()
    .reset_index()
    .rename(columns={"p_refund_count": "refund_count"})
    .sort_values(by=['refund_init_date']))
print(refund_final_status)



Empty DataFrame
Columns: [refund_init_date, refund_final_status_updated, refund_count]
Index: []


C:\Users\rajeshkumar.t\AppData\Local\Temp\ipykernel_9772\3756369509.py:5: FutureWarning: The behavior of 'isin' with dtype=datetime64[ns] and castable values (e.g. strings) is deprecated. In a future version, these will not be considered matching by isin. Explicitly cast to the appropriate dtype before calling isin instead.
  filtered_date = df[df['refund_init_date'].isin(['2026-01-10','2026-01-11','2026-01-12','2026-01-13'])]


In [7]:
df= df[(~df["marketplace_id"].isin(["HYPERLOCAL"])) &
                          ~df["is_shopsy_order"].isin(["true"])].copy()

within_24 = ["2. 0 To 4 Hrs", "3. 4 To 8 Hrs", "4. 8 To 24 Hrs"]
df["refund_init_completion_bucket_flag"]=df["refund_init_completion_bucket"].apply(
    lambda x: "within_24" if x in within_24 else "greater_24"
)

transaction_order = ['Completed', 'Failed', 'Cancelled', 'Stuck', 'CLONED']
df["refund_final_status_updated"] = pd.Categorical(df["refund_final_status_updated"], categories=transaction_order, ordered=True)

df["refund_init_date"] = pd.to_datetime(df["refund_init_date"], errors='coerce', format='%d-%m-%Y')

refund_final_status = (df.groupby(["refund_final_status_updated","refund_init_completion_bucket_flag", "refund_init_date"], observed=False)["p_refund_count"].sum().reset_index()
                .rename(columns={"p_refund_count": "refund_status_count"}))

refund_final_status = (
    df.groupby(["refund_final_status_updated", "refund_init_completion_bucket_flag", "refund_init_date"], observed=False)["p_refund_count"]
    .sum()
    .reset_index()
    .rename(columns={"p_refund_count": "refund_status_count"})
)
pivot_source = refund_final_status.pivot_table(index=["refund_final_status_updated","refund_init_completion_bucket_flag"],
                                                columns = "refund_init_date",
                                                values = "refund_status_count",
                                                aggfunc='sum',
                                                observed= False
                                              ).fillna(0)
                                              
pivot_source.columns=pd.to_datetime(pivot_source.columns, errors='coerce')
pivot_source = pivot_source.sort_index(axis=1)
pivot_source.columns = pivot_source.columns.strftime('%d-%m-%Y')
pivot_source.reset_index(inplace=True)

#print(pivot_source)

####-------------------------###

df["p_refund_count"] = pd.to_numeric(df["p_refund_count"], errors = 'coerce')
df["refund_init_date"] = pd.to_datetime(df["refund_init_date"], errors='coerce', format='%d-%m-%Y')


transaction_order5 = ['Completed', 'Failed', 'Cancelled', 'Stuck', 'CLONED']
df["refund_final_status_updated"] = pd.Categorical(df["refund_final_status_updated"], categories=transaction_order, ordered=True)

df["refund_init_date"] = pd.to_datetime(df["refund_init_date"], errors='coerce', format='%d-%m-%Y')


refund_final_status1 = (
    df.groupby(["refund_final_status_updated",  "refund_init_date"], observed=False)["p_refund_count"]
    .sum()
    .reset_index()
    .rename(columns={"p_refund_count": "refund_status_count1"})
)

pivot_source1 = refund_final_status1.pivot_table(index=["refund_final_status_updated"],
                                                columns = "refund_init_date",
                                                values = "refund_status_count1",
                                                aggfunc='sum',
                                                observed= False
                                              ).fillna(0)
                                              
pivot_source1.columns=pd.to_datetime(pivot_source1.columns, errors='coerce')
pivot_source1 = pivot_source1.sort_index(axis=1)
pivot_source1.columns = pivot_source1.columns.strftime('%d-%m-%Y')
pivot_source1.reset_index(inplace=True)

#print(pivot_source1)


##------------------##
##refund_reason_flag
df["refund_reason_group"] = df["refund_reason_flag"].apply(
    lambda x: "Cancel_RTO" if x in ["Cancellation", "RTO"] 
    else x if x in ["Return","Adonc"]
    else "Others"
)

df["p_refund_count"] = pd.to_numeric(df["p_refund_count"], errors = 'coerce')
df["refund_init_date"] = pd.to_datetime(df["refund_init_date"], errors='coerce', format='%d-%m-%Y')

transaction_order1 = ['Cancel_RTO', 'Return', 'Adonc', 'Others']
df["refund_reason_group"] = pd.Categorical(df["refund_reason_group"], categories=transaction_order1, ordered=True)

refund_reason_flag1 = (df.groupby(["refund_reason_group", "refund_init_date"], observed=False)["p_refund_count"].sum().reset_index()
                .rename(columns={"p_refund_count": "refund_reason_group_count"}))

pivot_source_reason = refund_reason_flag1.pivot_table(index="refund_reason_group",
                                                columns = "refund_init_date",
                                                values = "refund_reason_group_count",
                                                aggfunc='sum',
                                                observed=False     
                                                 ).fillna(0)
                                              
pivot_source_reason.columns=pd.to_datetime(pivot_source_reason.columns, errors='coerce')
pivot_source_reason = pivot_source_reason.sort_index(axis=1)
pivot_source_reason.columns = pivot_source_reason.columns.strftime('%d-%m-%Y')
pivot_source_reason.reset_index(inplace=True)

#print(pivot_source_reason)

##------------------##
##refund_reason_flag
df["payment_type_final_flag"] = df["payment_type_final"].apply(
    lambda x: "Pre-paid" if x in ["Pre-paid", "EGV/Wallet/Coin"] 
    else x 
)
transaction_order2 = ['Pre-paid', 'COD', 'Multi-payment', 'NA']
df["payment_type_final_flag"] = pd.Categorical(df["payment_type_final_flag"], categories=transaction_order2, ordered=True)

refund_summary_by_payment_type = (df.groupby(["payment_type_final_flag", "refund_init_date"], observed=False)["p_refund_count"].sum().reset_index()
                .rename(columns={"p_refund_count": "payment_type_final_count"}))

pivot_source_payment = refund_summary_by_payment_type.pivot_table(index="payment_type_final_flag",
                                                columns = "refund_init_date",
                                                values = "payment_type_final_count",
                                                aggfunc='sum',
                                                observed=False     
                                                 ).fillna(0)
                                              
pivot_source_payment.columns=pd.to_datetime(pivot_source_payment.columns, errors='coerce')
pivot_source_payment = pivot_source_payment.sort_index(axis=1)
pivot_source_payment.columns = pivot_source_payment.columns.strftime('%d-%m-%Y')
pivot_source_payment.reset_index(inplace=True)

##print(pivot_source_payment)

##------------------##
##ARN_issue_flag

transaction_order3 = ['ARN_GENERATED', 'ARN_NOT_GENERATED', 'ARN_NOT_APPLICABLE']
df["ARN_ISSUE_FLAG"] = pd.Categorical(df["ARN_ISSUE_FLAG"], categories=transaction_order3, ordered=True)

ARN_issue_flag_summary = (df.groupby(["ARN_ISSUE_FLAG", "refund_init_date"], observed=False)["p_refund_count"].sum().reset_index()
                .rename(columns={"p_refund_count": "ARN_ISSUE_count"}))

pivot_source_ARN = ARN_issue_flag_summary.pivot_table(index="ARN_ISSUE_FLAG",
                                                columns = "refund_init_date",
                                                values = "ARN_ISSUE_count",
                                                aggfunc='sum',
                                                observed=False     
                                                 ).fillna(0)
                                              
pivot_source_ARN.columns=pd.to_datetime(pivot_source_ARN.columns, errors='coerce')
pivot_source_ARN = pivot_source_ARN.sort_index(axis=1)
pivot_source_ARN.columns = pivot_source_ARN.columns.strftime('%d-%m-%Y')
pivot_source_ARN.reset_index(inplace=True)

##print(pivot_source_ARN)

##------------------##
##Refund_overall


Refund_overall_summary = (df.groupby(["refund_init_date"], observed=False)["p_refund_count"].sum().reset_index()
                .rename(columns={"p_refund_count": "Overall_refund_count"}))

Refund_overall_summary = Refund_overall_summary.sort_values("refund_init_date")
pivot_Refund_overall_summary=Refund_overall_summary.set_index("refund_init_date").T
                                              
pivot_Refund_overall_summary.columns=pd.to_datetime(pivot_Refund_overall_summary.columns, errors='coerce')
pivot_Refund_overall_summary = pivot_Refund_overall_summary.sort_index(axis=1)
pivot_Refund_overall_summary.columns = pivot_Refund_overall_summary.columns.strftime('%d-%m-%Y')
pivot_Refund_overall_summary.reset_index(inplace=True)

#print(pivot_Refund_overall_summary)



##---Void_Transactions-----

##Refund_overall


Refund_void_summary = (df.groupby(["void_flag", "refund_init_date"], observed=False)["p_refund_count"].sum().reset_index()
                .rename(columns={"p_refund_count": "refund_void_count"}))

pivot_void_summary = Refund_void_summary.pivot_table(index="void_flag",
                                                columns = "refund_init_date",
                                                values = "refund_void_count",
                                                aggfunc='sum',
                                                observed=False     
                                                 ).fillna(0)
                                              
pivot_void_summary.columns=pd.to_datetime(pivot_void_summary.columns, errors='coerce')
pivot_void_summary = pivot_void_summary.sort_index(axis=1)
pivot_void_summary.columns = pivot_void_summary.columns.strftime('%d-%m-%Y')
pivot_void_summary.reset_index(inplace=True)

##print(pivot_void_summary)

#####------------------------
##Refund_modes

emi_pg_ids =['cfa_juspay_dmi_emi',
    'cfa_juspay_fibe_emi',
    'cfa_juspay_abfl_emi',
    'cfa_juspay_chola_emi',
    'cfa_juspay_tvs_emi'
]

df["refund_mode"] = df.apply(
lambda row: 'fk_emi' if row['pg_id'] in emi_pg_ids else row['refund_mode'], axis=1)

transaction_order4 = ["UPI","UPI_COLLECT","UPI_INTENT","FK_UPI","PHONEPE","NETBANKING","NEFT","IMPS","RTGS","CREDIT_CARD","CREDIT_CARD_EMI","DEBIT_CARD","DYNAMIC_QR","COIN",
        "E_GIFT_VOUCHER","QC_EGV","EGV","EGV_WALLET","WALLET","FLIPKART_FINANCE","InvalidRefundMode","LOAN","SUPERPAY_LATER","SUPERPAY_IN_INSTALLMENT","fk_emi"]

df["refund_mode"] = df["refund_mode"].apply(
    lambda x: x if x in transaction_order4 else "Others"
)
category_list = transaction_order4 + ["Others"]
df["refund_mode"] = pd.Categorical(df["refund_mode"], categories=category_list, ordered=True)


refund_mode_flag_summary = (df.groupby(["refund_mode", "refund_init_date"], observed=False)["p_refund_count"].sum().reset_index()
                .rename(columns={"p_refund_count": "refund_mode_count"}))

pivot_source_refund_mode = refund_mode_flag_summary.pivot_table(index="refund_mode",
                                                columns = "refund_init_date",
                                                values = "refund_mode_count",
                                                aggfunc='sum',
                                                observed=False     
                                                 ).fillna(0)
                                              
pivot_source_refund_mode.columns=pd.to_datetime(pivot_source_refund_mode.columns, errors='coerce')
pivot_source_refund_mode = pivot_source_refund_mode.sort_index(axis=1)
pivot_source_refund_mode.columns = pivot_source_refund_mode.columns.strftime('%d-%m-%Y')
pivot_source_refund_mode.reset_index(inplace=True)

output_path = r"G:\My Drive\New_Working_Files\KNIME\CTU_2025\Payment_Adonc\Refund_tranaction_output.xlsx"
with pd.ExcelWriter(output_path, engine = 'openpyxl') as writer:
    pivot_source.to_excel(writer, sheet_name ='refund_final_status_updated', index=False)
    pivot_source1.to_excel(writer, sheet_name ='refund_final_status', index=False)
    pivot_source_reason.to_excel(writer, sheet_name ='refund_reason', index=False )
    pivot_source_payment.to_excel(writer, sheet_name ='refund_source_payment', index=False )
    pivot_source_ARN.to_excel(writer, sheet_name = 'refund_ARN_flag', index=False)
    pivot_void_summary.to_excel(writer, sheet_name = 'pivot_Refund_Void', index=False)
    pivot_source_refund_mode.to_excel(writer, sheet_name = 'refund_mode_ssi', index=False)
    pivot_Refund_overall_summary.to_excel(writer, sheet_name ='Overall_count, index =False')



print("Excel file successfully saved:",output_path)

Excel file successfully saved: G:\My Drive\New_Working_Files\KNIME\CTU_2025\Payment_Adonc\Refund_tranaction_output.xlsx


In [15]:
import pandas as pd

df['refund_init_date'] = pd.to_datetime(df['refund_init_date'], errors='coerce')

transaction_order4 = ["UPI_INTENT","COIN","COD","CREDIT","UPI_COLLECT","FK_UPI","EGV_WALLET",
                      "EMI","DEBIT","NETBANKING","EGV","DYNAMIC_QR","SUPERPAY_IN_INSTALLMENT",
                      "SUPERPAY_LATER","RTGS","PHONEPE"]

df["payment_instrument"] = df["payment_instrument"].apply(
    lambda x: x if x in transaction_order4 else "Others")

refund_status_group = {
    "UPI_INTENT": ["UPI_INTENT", "QC_EGV", "IMPS"],
    "COIN": ["COIN"],
    "COD": ["IMPS", "UPI", "QC_EGV", "NEFT"],
    "CREDIT": ["CREDIT_CARD", "QC_EGV"],
    "UPI_COLLECT": ["UPI_COLLECT", "QC_EGV"],
    "FK_UPI": ["FK_UPI"],
    "EGV_WALLET": ["EGV_WALLET"],
    "EMI": ["CREDIT_CARD_EMI"],
    "DEBIT": ["DEBIT_CARD", "QC_EGV"],
    "NETBANKING": ["NETBANKING", "QC_EGV"],
    "EGV": ["EGV"],
    "DYNAMIC_QR": ["DYNAMIC_QR"],
    "SUPERPAY_IN_INSTALLMENT": ["SUPERPAY_IN_INSTALLMENT"],
    "SUPERPAY_LATER": ["SUPERPAY_LATER"],
    "RTGS": ["RTGS"],
    "PHONEPE": ["IMPS", "PHONEPE"]
}

refund_mode_summary_payment = (
    df.groupby(["payment_instrument", "refund_mode", "refund_init_date"], observed=False)["p_refund_count"]
    .sum()
    .reset_index()
    .rename(columns={"p_refund_count": "payment_mode_count"})
)

refund_mode_summary_payment_pivot = refund_mode_summary_payment.pivot_table(
    index=["payment_instrument", "refund_mode"], 
    columns="refund_init_date",
    values="payment_mode_count",
    aggfunc="sum",
    fill_value=0
).reset_index()
refund_mode_summary_payment_pivot.columns.name= None

subtotal = (
    refund_mode_summary_payment.groupby(["payment_instrument","refund_init_date"])["payment_mode_count"]
    .sum()
    .reset_index()
)
    
subtotal = (
    refund_mode_summary_payment.groupby(["payment_instrument", "refund_init_date"])["payment_mode_count"]
    .sum()
    .reset_index()
)
subtotal["refund_mode"] = "Subtotal"
subtotal_pivot = subtotal.pivot_table(
    index=["payment_instrument","refund_mode"],
    columns = "refund_init_date",
    values="payment_mode_count",
    aggfunc = "sum",
    fill_value = 0
).reset_index()
subtotal_pivot.columns.name = None

final_df = pd.concat([refund_mode_summary_payment_pivot, subtotal_pivot], ignore_index=True)
final_df.sort_values(by=["payment_instrument", "refund_mode"], inplace=True)

final_df.columns = [
    col.strftime('%d-%m-%Y') if isinstance(col, pd.Timestamp) else col
    for col in final_df.columns
]
final_df["payment_instrument"] = pd.Categorical(
    final_df["payment_instrument"], categories= transaction_order4, ordered=True)
    
final_df.sort_values(by=["payment_instrument", "refund_mode"], inplace=True)

# Display result
#print(final_df.head(25))




C:\Users\rajeshkumar.t\AppData\Local\Temp\ipykernel_7244\1931929860.py:38: FutureWarning: The default value of observed=False is deprecated and will change to observed=True in a future version of pandas. Specify observed=False to silence this warning and retain the current behavior
  refund_mode_summary_payment_pivot = refund_mode_summary_payment.pivot_table(


In [16]:
ordering_map = pd.DataFrame([
    {"payment_instrument": "UPI_INTENT", "refund_mode": "Subtotal", "ordering": 1},
    {"payment_instrument": "UPI_INTENT", "refund_mode": "UPI_INTENT", "ordering": 2},
    {"payment_instrument": "UPI_INTENT", "refund_mode": "QC_EGV", "ordering": 3},
    {"payment_instrument": "UPI_INTENT", "refund_mode": "IMPS", "ordering": 4},
    {"payment_instrument": "COIN", "refund_mode": "Subtotal", "ordering": 5},
    {"payment_instrument": "COIN", "refund_mode": "COIN", "ordering": 6},
    {"payment_instrument": "COD", "refund_mode": "Subtotal", "ordering": 7},
    {"payment_instrument": "COD", "refund_mode": "IMPS", "ordering": 8},
    {"payment_instrument": "COD", "refund_mode": "UPI", "ordering": 9},
    {"payment_instrument": "COD", "refund_mode": "QC_EGV", "ordering": 10},
    {"payment_instrument": "COD", "refund_mode": "NEFT", "ordering": 11},
    {"payment_instrument": "CREDIT", "refund_mode": "Subtotal", "ordering": 12},
    {"payment_instrument": "CREDIT", "refund_mode": "CREDIT_CARD", "ordering": 13},
    {"payment_instrument": "CREDIT", "refund_mode": "QC_EGV", "ordering": 14},
    {"payment_instrument": "UPI_COLLECT", "refund_mode": "Subtotal", "ordering": 15},
    {"payment_instrument": "UPI_COLLECT", "refund_mode": "UPI_COLLECT", "ordering": 16},
    {"payment_instrument": "UPI_COLLECT", "refund_mode": "QC_EGV", "ordering": 17},
    {"payment_instrument": "FK_UPI", "refund_mode": "Subtotal", "ordering": 18},
    {"payment_instrument": "FK_UPI", "refund_mode": "FK_UPI", "ordering": 19},
    {"payment_instrument": "EGV_WALLET", "refund_mode": "Subtotal", "ordering": 20},
    {"payment_instrument": "EGV_WALLET", "refund_mode": "EGV_WALLET", "ordering": 21},
    {"payment_instrument": "EMI", "refund_mode": "Subtotal", "ordering": 22},
    {"payment_instrument": "EMI", "refund_mode": "CREDIT_CARD_EMI", "ordering": 23},
    {"payment_instrument": "DEBIT", "refund_mode": "Subtotal", "ordering": 24},
    {"payment_instrument": "DEBIT", "refund_mode": "DEBIT_CARD", "ordering": 25},
    {"payment_instrument": "DEBIT", "refund_mode": "QC_EGV", "ordering": 26},
    {"payment_instrument": "NETBANKING", "refund_mode": "NETBANKING", "ordering": 27},
    {"payment_instrument": "NETBANKING", "refund_mode": "QC_EGV", "ordering": 28},
    {"payment_instrument": "EGV", "refund_mode": "Subtotal", "ordering": 29},
    {"payment_instrument": "EGV", "refund_mode": "EGV", "ordering": 30},
    {"payment_instrument": "DYNAMIC_QR", "refund_mode": "Subtotal", "ordering": 31},
    {"payment_instrument": "DYNAMIC_QR", "refund_mode": "DYNAMIC_QR", "ordering": 32},
    {"payment_instrument": "SUPERPAY_IN_INSTALLMENT", "refund_mode": "Subtotal", "ordering": 33},
    {"payment_instrument": "SUPERPAY_IN_INSTALLMENT", "refund_mode": "SUPERPAY_IN_INSTALLMENT", "ordering": 34},
    {"payment_instrument": "SUPERPAY_LATER", "refund_mode": "Subtotal", "ordering": 35},
    {"payment_instrument": "SUPERPAY_LATER", "refund_mode": "SUPERPAY_LATER", "ordering": 36},
    {"payment_instrument": "RTGS", "refund_mode": "Subtotal", "ordering": 37},
    {"payment_instrument": "RTGS", "refund_mode": "RTGS", "ordering": 38},
    {"payment_instrument": "PHONEPE", "refund_mode": "Subtotal", "ordering": 39},
    {"payment_instrument": "PHONEPE", "refund_mode": "IMPS", "ordering": 40},
    {"payment_instrument": "PHONEPE", "refund_mode": "PHONEPE", "ordering": 41},
])

final_df = final_df.merge(ordering_map, on=["payment_instrument", "refund_mode"], how="left")
final_df.sort_values(by=["ordering"], inplace=True)
final_df.drop(columns="ordering", inplace=True) 

In [17]:
output_path = r"G:\My Drive\New_Working_Files\KNIME\CTU_2025\Payment_Adonc\Refund_payment_modes.xlsx"
with pd.ExcelWriter(output_path, engine = 'openpyxl') as writer:
    final_df.to_excel(writer, sheet_name ='refund_final_payment_mode', index=False)


print("Excel file successfully saved:", output_path)

Excel file successfully saved: G:\My Drive\New_Working_Files\KNIME\CTU_2025\Payment_Adonc\Refund_payment_modes.xlsx


In [20]:
raw_df = pd.read_csv(transaction_path, low_memory=False)
print(raw_df.columns.tolist())
print(raw_df["refund_init_date"].head(10))

['refund_year', 'refund_month', 'refund_week', 'refund_init_date', 'refund_complete_date', 'refund_status', 'refund_final_status_updated', 'mr_refund_flag', 'refund_mode', 'auth_state', 'payment_mode', 'instrument_type', 'payment_instrument', 'ref_amount_flag', 'transaction_source', 'pg_id', 'promise_sla_refund_bucket', 'refund_init_completion_bucket', 'overall_ageing_Stuck', 'refund_complition_bucket', 'ARN_ISSUE_FLAG', 'void_flag', 'refund_reason', 'refund_reason_flag', 'flipkart_emi_flag', 'payment_type_final', 'cx_count', 'rf_orders', 'p_refund_count', 'c_refund_count', 'total_refund_amount', 'is_shopsy_order', 'marketplace_id']
0    2025-10-01
1    2025-10-01
2    2025-10-01
3    2025-10-01
4    2025-10-01
5    2025-10-01
6    2025-10-01
7    2025-10-01
8    2025-10-01
9    2025-10-01
Name: refund_init_date, dtype: object


In [8]:
from pandas import MultiIndex

df = df[(~df["marketplace_id"].isin(["HYPERLOCAL"])) & (~df["is_shopsy_order"].isin(["true"]))].copy()
print("Rows after initial filter:", len(df))

# Step 2: Clean
df["p_refund_count"] = pd.to_numeric(df["p_refund_count"], errors='coerce')

# Step 3: Define valid modes
refund_status_group = {
    'Completed': ['UPI_INTENT', 'CREDIT_CARD', 'COIN', 'IMPS', 'QC_EGV'],
    'Failed': ['IMPS', 'NEFT', 'CREDIT_CARD', 'UPI', 'UPI_INTENT'],
    'Cancelled': ['IMPS', 'NEFT', 'UPI_INTENT', 'CREDIT_CARD_EMI', 'FK_UPI'],
    'Stuck': ['IMPS', 'UPI', 'NEFT', 'UPI_INTENT', 'CREDIT_CARD']
}
refund_status_valid = [mode for group in refund_status_group.values() for mode in group]

# Step 4: Filter refund_mode
#df['refund_mode'] = df['refund_mode'].astype(str).str.upper()
#df = df[df['refund_mode'].isin(refund_status_valid)]
#print("Unique refund_mode values:", df['refund_mode'].unique())

# Step 5: Filter refund_final_status_updated
#valid_statuses = list(refund_status_group.keys())
#df = df[df['refund_final_status_updated'].isin(valid_statuses)]
#print("Unique refund_final_status_updated values:", df['refund_final_status_updated'].unique())

df['refund_mode'] =df['refund_mode'].astype(str).str.upper()
valid_pairs = set(
    (status, mode)
    for status, modes in refund_status_group.items()
    for mode in modes
)
df= df[df.apply(lambda x: (x['refund_final_status_updated'], x['refund_mode']) in valid_pairs, axis=1)]

# Step 6: Group
refund_mode_status_latest = (
    df.groupby(['refund_init_date', 'refund_mode', 'refund_final_status_updated'], observed=False)['p_refund_count']
    .sum()
    .reset_index()
    .rename(columns={"p_refund_count": "Refund_mode_count"})
)
print("Grouped rows:", len(refund_mode_status_latest))

# Step 7: Pivot
pivot_refund_mode_status_latest = refund_mode_status_latest.pivot_table(
    index=[ "refund_final_status_updated","refund_mode"],
    columns="refund_init_date",
    values="Refund_mode_count",
    aggfunc="sum",
    observed=False
).fillna(0)
print("Pivot shape:", pivot_refund_mode_status_latest.shape)

# Step 8: Format
pivot_refund_mode_status_latest.columns = pd.to_datetime(pivot_refund_mode_status_latest.columns, errors='coerce').strftime('%d-%m-%Y')
pivot_refund_mode_status_latest.reset_index(inplace=True)

# Step 9: Filter zero rows
#pivot_refund_mode_status_latest = pivot_refund_mode_status_latest[
#    (pivot_refund_mode_status_latest.drop(columns=[ 'refund_final_status_updated','refund_mode']) != 0).any(axis=1)
#]

expected_pairs = [
    (status,mode)
    for status, modes in refund_status_group.items()
    for mode in modes
]

expected_index = MultiIndex.from_tuples(expected_pairs, names=['refund_final_status_updated','refund_mode'])
pivot_refund_mode_status_latest.set_index(['refund_final_status_updated','refund_mode'], inplace=True)
pivot_refund_mode_status_latest = pivot_refund_mode_status_latest.reindex(expected_index, fill_value=0)
pivot_refund_mode_status_latest.reset_index(inplace=True)

# Step 10: Export
output_path = r"G:\My Drive\New_Working_Files\KNIME\CTU_2025\Payment_Adonc\refund_mode_latest_status.xlsx"
with pd.ExcelWriter(output_path, engine='openpyxl') as writer:
    pivot_refund_mode_status_latest.to_excel(writer, sheet_name='pivot_refund_mode_status', index=False)

print("Excel File Successfully saved:", output_path)


Rows after initial filter: 315673
Grouped rows: 1170
Pivot shape: (45, 26)
Excel File Successfully saved: G:\My Drive\New_Working_Files\KNIME\CTU_2025\Payment_Adonc\refund_mode_latest_status.xlsx


In [47]:
df = df[(~df["marketplace_id"].isin(["HYPERLOCAL"])) & (~df["is_shopsy_order"].isin(["true"]))].copy()
print(df)

Empty DataFrame
Columns: [refund_year, refund_month, refund_week, refund_init_date, refund_complete_date, refund_status, refund_final_status_updated, mr_refund_flag, refund_mode, auth_state, payment_mode, instrument_type, payment_instrument, ref_amount_flag, transaction_source, pg_id, promise_sla_refund_bucket, refund_init_completion_bucket, overall_ageing_Stuck, refund_complition_bucket, ARN_ISSUE_FLAG, void_flag, refund_reason, refund_reason_flag, flipkart_emi_flag, payment_type_final, cx_count, rf_orders, p_refund_count, c_refund_count, total_refund_amount, is_shopsy_order, marketplace_id, refund_init_completion_bucket_flag, refund_reason_group, payment_type_final_flag]
Index: []

[0 rows x 36 columns]


In [38]:
import pandas as pd

# Original mapping
refund_mode_map = {
    "UPI_INTENT": ["UPI_INTENT", "QC_EGV", "IMPS"],
    "COIN": ["COIN"],
    "COD": ["IMPS", "UPI", "QC_EGV", "NEFT"],
    "CREDIT": ["CREDIT_CARD", "QC_EGV"],
    "UPI_COLLECT": ["UPI_COLLECT", "QC_EGV"],
    "FK_UPI": ["FK_UPI"],
    "EGV_WALLET": ["EGV_WALLET"],
    "EMI": ["CREDIT_CARD_EMI"],
    "DEBIT": ["DEBIT_CARD", "QC_EGV"],
    "NETBANKING": ["NETBANKING", "QC_EGV"],
    "EGV": ["EGV"],
    "DYNAMIC_QR": ["DYNAMIC_QR"],
    "SUPERPAY_IN_INSTALLMENT": ["SUPERPAY_IN_INSTALLMENT"],
    "SUPERPAY_LATER": ["SUPERPAY_LATER"],
    "RTGS": ["RTGS"],
    "PHONEPE": ["IMPS", "PHONEPE"]
}

# Create MultiIndex from mapping
tuples = [(instrument, mode) for instrument, modes in refund_mode_map.items() for mode in modes]
multi_index = pd.MultiIndex.from_tuples(tuples, names=["payment_instrument", "refund_mode"])

# Create empty DataFrame with MultiIndex
refund_mode_df = pd.DataFrame(index=multi_index).reset_index()

print(refund_mode_df.head(50))


         payment_instrument              refund_mode
0                UPI_INTENT               UPI_INTENT
1                UPI_INTENT                   QC_EGV
2                UPI_INTENT                     IMPS
3                      COIN                     COIN
4                       COD                     IMPS
5                       COD                      UPI
6                       COD                   QC_EGV
7                       COD                     NEFT
8                    CREDIT              CREDIT_CARD
9                    CREDIT                   QC_EGV
10              UPI_COLLECT              UPI_COLLECT
11              UPI_COLLECT                   QC_EGV
12                   FK_UPI                   FK_UPI
13               EGV_WALLET               EGV_WALLET
14                      EMI          CREDIT_CARD_EMI
15                    DEBIT               DEBIT_CARD
16                    DEBIT                   QC_EGV
17               NETBANKING               NETB